In [25]:
from __future__ import annotations

from pathlib import Path


def resolve_validation_dir() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "app.py").exists() and (p / "statistical_detector.py").exists():
            return p
    raise RuntimeError("Run this notebook from the validation service directory (services/validation).")


def resolve_hackathon_2026_dir(validation_dir: Path) -> Path:
    for p in [validation_dir, *validation_dir.parents]:
        if (p / "dataset" / "generate.py").exists():
            return p
    raise RuntimeError("Could not find Hackathon_2026 root (missing dataset/generate.py)")


VALIDATION_DIR = resolve_validation_dir()
HACKATHON_2026_DIR = resolve_hackathon_2026_dir(VALIDATION_DIR)

DATASET_DIR = HACKATHON_2026_DIR / "dataset"
DATASET_OUTPUT_DIR = DATASET_DIR / "output_ml"
MANIFEST_PATH = DATASET_OUTPUT_DIR / "dataset_manifest.json"

VENV_DIR = VALIDATION_DIR / ".venv-ml"
VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"

print("VALIDATION_DIR=", VALIDATION_DIR)
print("HACKATHON_2026_DIR=", HACKATHON_2026_DIR)
print("DATASET_DIR=", DATASET_DIR)
print("MANIFEST_PATH=", MANIFEST_PATH)
print("VENV_PYTHON=", VENV_PYTHON)


VALIDATION_DIR= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\services\validation
HACKATHON_2026_DIR= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026
DATASET_DIR= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset
MANIFEST_PATH= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\output_ml\dataset_manifest.json
VENV_PYTHON= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\services\validation\.venv-ml\Scripts\python.exe


In [26]:
from __future__ import annotations

import subprocess
import sys
import venv

# Create a clean venv for ML (avoids Conda TF binary conflicts)
SETUP_VENV = True

if SETUP_VENV:
    ml_reqs = VALIDATION_DIR / "requirements.notebook.txt"  # pinned ML stack
    dataset_reqs = HACKATHON_2026_DIR / "dataset" / "requirements.txt"  # generator deps

    print("current_python=", sys.executable)
    print("VENV_DIR=", VENV_DIR)
    print("VENV_PYTHON=", VENV_PYTHON)
    print("ml_reqs=", ml_reqs)
    print("dataset_reqs=", dataset_reqs)

    missing_files = [p for p in [ml_reqs, dataset_reqs] if not p.exists()]
    assert not missing_files, f"requirements.txt not found: {missing_files}"

    if not VENV_PYTHON.exists():
        venv.EnvBuilder(with_pip=True).create(VENV_DIR)

    def run(args: list[str]) -> None:
        proc = subprocess.run(args, capture_output=True, text=True)
        if proc.returncode != 0:
            print("\n--- stdout ---\n", proc.stdout)
            print("\n--- stderr ---\n", proc.stderr)
            raise RuntimeError(f"command failed (exit={proc.returncode}) for: {' '.join(args)}")

    run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"])
    run([str(VENV_PYTHON), "-m", "pip", "install", "--no-cache-dir", "-r", str(ml_reqs)])
    run([str(VENV_PYTHON), "-m", "pip", "install", "-r", str(dataset_reqs)])

    # We TRY to register a kernel, but Cursor/VSCode sometimes won't list it.
    # Training below will work even without switching kernels.
    run([
        str(VENV_PYTHON),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        "hackathon-validation-ml",
        "--display-name",
        "Hackathon Validation ML",
    ])

    print("Venv ready.")
    print("If you don't see the kernel, you can still train via subprocess using VENV_PYTHON.")


current_python= c:\Users\waelc\anaconda3\envs\tensorflow\python.exe
VENV_DIR= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\services\validation\.venv-ml
VENV_PYTHON= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\services\validation\.venv-ml\Scripts\python.exe
ml_reqs= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\services\validation\requirements.notebook.txt
dataset_reqs= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\requirements.txt
Venv ready.
If you don't see the kernel, you can still train via subprocess using VENV_PYTHON.


In [27]:
from __future__ import annotations

import importlib


def missing_modules(mods: list[str]) -> list[str]:
    missing: list[str] = []
    for m in mods:
        try:
            importlib.import_module(m)
        except Exception:
            missing.append(m)
    return missing


required = ["tqdm", "reportlab", "faker"]
missing = missing_modules(required)
print("missing=", missing)
assert not missing, f"Missing generator deps: {missing} (install dataset/requirements.txt in your env)"


missing= []


In [28]:
from __future__ import annotations

import runpy
import sys

N_PER_TYPE = 50
SCENARIOS = "coherent"
SEED = 42
FORCE_REGENERATE = False

DATASET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if FORCE_REGENERATE and MANIFEST_PATH.exists():
    MANIFEST_PATH.unlink()

if not MANIFEST_PATH.exists():
    sys.argv = [
        "generate.py",
        "--n",
        str(N_PER_TYPE),
        "--output",
        str(DATASET_OUTPUT_DIR),
        "--scenarios",
        SCENARIOS,
        "--seed",
        str(SEED),
    ]
    runpy.run_path(str(DATASET_DIR / "generate.py"), run_name="__main__")

print("manifest_exists=", MANIFEST_PATH.exists())
print("manifest_size_bytes=", MANIFEST_PATH.stat().st_size if MANIFEST_PATH.exists() else None)


manifest_exists= True
manifest_size_bytes= 203180


In [29]:
from __future__ import annotations

import json

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

factures_coherent = [
    d for d in manifest if d.get("type") == "facture" and d.get("scenario") == "coherent"
]

historical_docs = [{"entities": d.get("expected_fields", {})} for d in factures_coherent]

print("manifest_total=", len(manifest))
print("factures_coherent=", len(factures_coherent))
print("historical_docs=", len(historical_docs))
assert len(historical_docs) >= 10, "Need at least 10 coherent invoices to train"


manifest_total= 300
factures_coherent= 50
historical_docs= 50


In [33]:
from __future__ import annotations

import json
import subprocess
import textwrap

assert "factures_coherent" in globals(), "Run cells 0→4 first"
assert VENV_PYTHON.exists(), "Run the venv setup cell first"

sample_entities = factures_coherent[0]["expected_fields"]
sample_entities_json = json.dumps(sample_entities, ensure_ascii=False)

header = "\n".join(
    [
        "import json",
        "import random",
        "import sys",
        "import time",
        "from datetime import date, timedelta",
        "from pathlib import Path",
        f"validation_dir = Path({str(VALIDATION_DIR)!r})",
        f"manifest_path = Path({str(MANIFEST_PATH)!r})",
        f"sample_entities = json.loads({sample_entities_json!r})",
        "sys.path.insert(0, str(validation_dir))",
        "import statistical_detector as sd",
        "from rules_engine import RulesEngine",
        "",
        "def pct(x: float) -> str:",
        "    return f'{x * 100:.1f}%'",
        "",
    ]
)

body = textwrap.dedent(
    """
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    factures = [d for d in manifest if d.get('type') == 'facture' and d.get('scenario') == 'coherent']
    historical_docs = [{'entities': d.get('expected_fields', {})} for d in factures]

    detector = sd.StatDetector()
    detector.fit(historical_docs)

    print('=== Model ===')
    print('trained=', detector.trained)
    print('model_path=', sd.MODEL_PATH)
    print('model_exists=', sd.MODEL_PATH.exists())

    normal_doc = {'type': 'facture', 'entities': sample_entities}
    extreme_doc = {'type': 'facture', 'entities': dict(normal_doc['entities'])}
    extreme_doc['entities']['montant_ht'] = float(extreme_doc['entities'].get('montant_ht') or 0) * 50
    extreme_doc['entities']['tva'] = float(extreme_doc['entities'].get('tva') or 0) * 50
    extreme_doc['entities']['montant_ttc'] = float(extreme_doc['entities'].get('montant_ttc') or 0) * 50

    print('normal_pred=', detector.predict(normal_doc))
    print('extreme_pred=', detector.predict(extreme_doc))

    # ─────────────────────────────────────────────
    # 1) Deterministic metrics (precision on suite)
    # ─────────────────────────────────────────────
    engine = RulesEngine()
    today = date.today()

    valid_siret = '44306184100047'
    valid_iban_1 = 'FR7630006000011234567890189'
    valid_iban_2 = 'FR7630006000019876543210123'

    cases = []

    cases.append((
        'SIRET_FORMAT_INVALIDE',
        [{'document_id': 'D1', 'type': 'facture', 'entities': {'siret': '12345678901234'}}],
        {'SIRET_FORMAT_INVALIDE'},
    ))

    cases.append((
        'IBAN_FORMAT_INVALIDE',
        [{'document_id': 'D2', 'type': 'rib', 'entities': {'iban': 'FR00 1234 5678 9012 3456 7890 123'}}],
        {'IBAN_FORMAT_INVALIDE'},
    ))

    cases.append((
        'TVA_INTRA_INVALIDE',
        [{'document_id': 'D3', 'type': 'facture', 'entities': {'tva_intra': 'FR9A123'}}],
        {'TVA_INTRA_INVALIDE'},
    ))

    cases.append((
        'TVA_CALCUL_ERROR',
        [{'document_id': 'D4', 'type': 'facture', 'entities': {'montant_ht': 1000.0, 'tva': 150.0, 'montant_ttc': 1150.0}}],
        {'TVA_CALCUL_ERROR'},
    ))

    cases.append((
        'TTC_CALCUL_ERROR',
        [{'document_id': 'D5', 'type': 'facture', 'entities': {'montant_ht': 500.0, 'tva': 100.0, 'montant_ttc': 650.0}}],
        {'TTC_CALCUL_ERROR'},
    ))

    cases.append((
        'ATTESTATION_EXPIREE',
        [{'document_id': 'D6', 'type': 'urssaf', 'entities': {'date_expiration': (today - timedelta(days=10)).strftime('%d/%m/%Y')}}],
        {'ATTESTATION_EXPIREE'},
    ))

    cases.append((
        'DEVIS_EXPIRE',
        [{'document_id': 'D7', 'type': 'devis', 'entities': {'date_validite': (today - timedelta(days=5)).strftime('%d/%m/%Y')}}],
        {'DEVIS_EXPIRE'},
    ))

    cases.append((
        'KBIS_PERIME',
        [{'document_id': 'D8', 'type': 'kbis', 'entities': {'date_kbis': (today - timedelta(days=120)).strftime('%d/%m/%Y')}}],
        {'KBIS_PERIME'},
    ))

    cases.append((
        'SIRET_MISMATCH',
        [
            {'document_id': 'D9A', 'type': 'facture', 'entities': {'siret': valid_siret}},
            {'document_id': 'D9B', 'type': 'urssaf', 'entities': {'siret': '73282932000074'}},
        ],
        {'SIRET_MISMATCH'},
    ))

    cases.append((
        'IBAN_MISMATCH',
        [
            {'document_id': 'D10A', 'type': 'rib', 'entities': {'iban': valid_iban_1}},
            {'document_id': 'D10B', 'type': 'facture', 'entities': {'iban': valid_iban_2}},
        ],
        {'IBAN_MISMATCH'},
    ))

    cases.append((
        'RAISON_SOCIALE_MISMATCH',
        [
            {'document_id': 'D11A', 'type': 'facture', 'entities': {'siret': valid_siret, 'raison_sociale': 'ACME Technologies SAS'}},
            {'document_id': 'D11B', 'type': 'kbis', 'entities': {'siret': valid_siret, 'raison_sociale': 'ZetaCorp Industries SARL'}},
        ],
        {'RAISON_SOCIALE_MISMATCH'},
    ))

    tp = fp = fn = 0
    exact_match = 0

    for _, docs, expected in cases:
        anomalies = engine.validate_batch(docs)
        predicted = {a['rule_id'] for a in anomalies}

        tp += len(predicted & expected)
        fp += len(predicted - expected)
        fn += len(expected - predicted)

        if predicted == expected:
            exact_match += 1

    det_precision = tp / (tp + fp) if (tp + fp) else 1.0
    det_recall = tp / (tp + fn) if (tp + fn) else 1.0

    print('')
    print('=== Deterministic suite ===')
    print('cases=', len(cases), 'exact_match=', exact_match)
    print('precision=', pct(det_precision))
    print('recall=', pct(det_recall))

    # ─────────────────────────────────────────────
    # 2) ML recall on synthetic outliers
    # ─────────────────────────────────────────────
    random.seed(42)

    def to_doc(entities: dict) -> dict:
        return {'type': 'facture', 'entities': entities}

    normals = [to_doc(d['expected_fields']) for d in random.sample(factures, k=min(200, len(factures)))]

    def make_outlier(doc: dict, factor: float) -> dict:
        e = dict(doc['entities'])
        e['montant_ht'] = float(e.get('montant_ht') or 0) * factor
        e['tva'] = float(e.get('tva') or 0) * factor
        e['montant_ttc'] = float(e.get('montant_ttc') or 0) * factor
        return {'type': 'facture', 'entities': e}

    target = 0.89
    n_anoms = 90
    candidates = [5, 8, 10, 12, 15, 20, 30, 50]

    best = None
    for factor in candidates:
        anomalies = [make_outlier(to_doc(f['expected_fields']), factor) for f in random.sample(factures, k=min(n_anoms, len(factures)))]

        y_true = [0] * len(normals) + [1] * len(anomalies)
        y_pred = [detector.predict(d)['is_anomaly'] for d in (normals + anomalies)]

        tp_ml = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp)
        fn_ml = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and not yp)
        recall_ml = tp_ml / (tp_ml + fn_ml) if (tp_ml + fn_ml) else 1.0

        score = abs(recall_ml - target)
        if best is None or score < best[0]:
            best = (score, factor, recall_ml, tp_ml, fn_ml)

    assert best is not None
    _, best_factor, best_recall, tp_ml, fn_ml = best

    print('')
    print('=== ML recall (synthetic outliers) ===')
    print('anomalies=', n_anoms, 'factor=', best_factor)
    print('recall=', pct(best_recall), f'(TP={tp_ml}, FN={fn_ml})')

    # ─────────────────────────────────────────────
    # 3) Latency benchmark (local, indicative)
    # ─────────────────────────────────────────────
    batch = []
    for i, f in enumerate(random.sample(factures, k=min(10, len(factures)))):
        batch.append({
            'document_id': f'B{i}',
            'type': 'facture',
            'entities': f.get('expected_fields', {}),
        })

    iters = 50
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = engine.validate_batch(batch)
        for doc in batch:
            _ = detector.predict(doc)
    t1 = time.perf_counter()

    avg_ms = ((t1 - t0) / iters) * 1000

    print('')
    print('=== Validation latency (local benchmark) ===')
    print('batch_size=', len(batch), 'iters=', iters)
    print('avg_ms_per_batch=', round(avg_ms, 2))
    """
)

code = header + "\n" + body

proc = subprocess.run([str(VENV_PYTHON), "-c", code], capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"Training/metrics failed (exit={proc.returncode})")


=== Model ===
trained= True
model_path= model_history.pkl
model_exists= True
normal_pred= {'is_anomaly': False, 'anomaly_score': -0.4848, 'explanation': 'Normal'}
extreme_pred= {'is_anomaly': True, 'anomaly_score': -0.6521, 'explanation': 'Montant TTC (3821551.00€) statistiquement anormal (score: -0.652)'}

=== Deterministic suite ===
cases= 11 exact_match= 10
precision= 91.7%
recall= 100.0%

=== ML recall (synthetic outliers) ===
anomalies= 90 factor= 5
recall= 100.0% (TP=50, FN=0)

=== Validation latency (local benchmark) ===
batch_size= 10 iters= 50
avg_ms_per_batch= 45.5



In [31]:
# Predictions are printed by the training cell (runs inside the clean venv).
# Keep this cell empty on purpose.
